# ReAct Agent Test

In [1]:
# 대화 세션 초기화 — 새로운 thread_id로 기존 대화 기록 없이 시작
import uuid
from dotenv import load_dotenv

load_dotenv()

from Agent.graph import react_agent

thread_id = f"notebook-{uuid.uuid4().hex[:8]}"
config = {"configurable": {"thread_id": thread_id}}
print(f"세션 초기화 완료: {thread_id}")

/home/shin/Study/ReActAgent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


세션 초기화 완료: notebook-e7dfae55


In [2]:
def ask(query: str):
    """Agent에 질문하고 이번 턴의 결과만 출력합니다."""
    result = react_agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=config,
    )
    # 이번 턴의 새 메시지만 출력 (마지막 user 메시지부터)
    msgs = result["messages"]
    start = len(msgs)
    for i in range(len(msgs) - 1, -1, -1):
        if msgs[i].type == "human":
            start = i
            break
    for msg in msgs[start:]:
        print(f"[{msg.type}] {msg.content}\n")

In [3]:
ask("사용 가능한 도구 목록 보여줘")

[human] 사용 가능한 도구 목록 보여줘

[ai] 현재 사용 가능한 도구 목록입니다:

## 📁 파일시스템 도구
- **ls** - 디렉토리 내 파일 목록 조회
- **read_file** - 파일 읽기 (페이지네이션 지원)
- **write_file** - 새 파일 작성
- **edit_file** - 기존 파일 수정
- **glob** - 패턴으로 파일 찾기 (예: `**/*.py`)
- **grep** - 파일 내 텍스트 검색

## 🔍 검색 및 정보 도구
- **web_search** - 인터넷 최신 정보 검색 (뉴스, 날씨, 실시간 정보)
- **kifrs_search** - K-IFRS(한국채택국제회계기준) 기준서 검색
  - Parent-Child 계층 검색으로 관련 문단과 형제 문단 함께 반환
  - standard_id로 특정 기준서 지정 가능 (예: '1115', '1037')

## 🧮 계산 도구
- **calculator** - 수학 계산 수행 (사칙연산, 거듭제곱, 나머지 등)

## 🤖 에이전트 도구
- **task** - 복잡한 작업을 위한 서브에이전트 실행
  - **general-purpose**: 범용 에이전트 (모든 도구 접근 가능)
  - **kifrs-expert**: K-IFRS 기준서 개별 분석 전문 에이전트

## 📋 작업 관리 도구
- **write_todos** - 복잡한 작업의 할일 목록 생성 및 관리

## 💡 전문 스킬
- **k-ifrs-reasoning**: 복합 K-IFRS 회계 질문 체계적 분석
- **unit-converter**: 단위 변환 가이드

어떤 작업을 도와드릴까요?



## 1. Calculator

In [15]:
ask("123 * 456 + 789 계산해줘")

[human] 123 * 456 + 789 계산해줘

[ai] [{'id': 'toolu_011h7ALvPvP4pZDU86JHcT1m', 'caller': {'type': 'direct'}, 'input': {'expression': '123 * 456 + 789'}, 'name': 'calculator', 'type': 'tool_use'}]

[tool] 123 * 456 + 789 = 56877

[ai] 123 × 456 + 789 = **56,877**입니다.



## 2. Web Search

In [16]:
ask("오늘 주요 뉴스 알려줘")

[human] 오늘 주요 뉴스 알려줘

[ai] [{'id': 'toolu_015oQG6Bxuk2Raj7tUfPUixM', 'caller': {'type': 'direct'}, 'input': {'query': '오늘 주요 뉴스', 'topic': 'news', 'max_results': 5}, 'name': 'web_search', 'type': 'tool_use'}]

[tool] **Man dies in car hauler accident - CBS News**
Image 23: Showers today warming through the weekend ### Showers today warming through the weekend 03:06 2h agoImage 24: Senate fails to pass act limiting Trump's war powers ### Senate fails to pass act limiting Trump's war powers 02:15 8h agoImage 25: Highlands Detroit announces closing date in RenCen ### Highlands Detroit announces closing date in RenCen 02:29 8h agoImage 26: Detroit officials host community safety forum ### Detroit officials host community safety forum 00:53 8h agoImage 27: Metro Detroit leaders urge Congress to oppose Iran war ### Metro Detroit leaders urge Congress to oppose Iran war 02:18 8h agoImage 28: Here comes the rain ### Here comes the rain 02:50 12h agoImage 29: Ajinomoto recalls 37 million pounds

## 3. Skills

In [4]:
ask("사용 가능한 스킬 목록 보여줘")

[human] 사용 가능한 스킬 목록 보여줘

[ai] 현재 사용 가능한 스킬 목록입니다:

## 📚 전문 스킬

### 1. **k-ifrs-reasoning**
- **설명**: 복합 K-IFRS 회계 질문을 체계적으로 분석하는 프레임워크. 다중 기준서 교차분석이 필요할 때 사용합니다.
- **사용 도구**: kifrs_search, write_todos, task, write_file, read_file
- **위치**: `/home/shin/Study/ReActAgent/skills/k-ifrs-reasoning/SKILL.md`
- **적용 사례**:
  - 여러 기준서를 교차 참조해야 하는 복잡한 회계 질문
  - 자산/부채/수익 인식 시점 판단
  - 측정 및 공정가치 평가 문제

### 2. **unit-converter**
- **설명**: 자주 사용되는 단위 변환 가이드
- **위치**: `/home/shin/Study/ReActAgent/skills/unit-converter/SKILL.md`
- **적용 사례**:
  - 길이, 무게, 온도 등 단위 변환
  - 통화 단위 변환
  - 시간 단위 변환

---

## 📖 스킬 사용 방법

스킬은 **점진적 공개(Progressive Disclosure)** 방식으로 작동합니다:

1. **스킬 인식**: 사용자 요청이 스킬 설명과 일치하는지 확인
2. **전체 지침 읽기**: 위 경로의 SKILL.md 파일 읽기
3. **지침 따르기**: 단계별 워크플로우, 모범 사례, 예제 활용
4. **지원 파일 접근**: 스킬에 포함된 스크립트나 참조 문서 사용

어떤 스킬을 사용해보시겠어요?



## 4. K-IFRS Search

In [8]:
ask("(주)라비티는 전기에 액면가 5000원의 100주의 주식을 주당 6000원에 발행하여 설립되었다. 당기 2월 1일, (주)라비티는 추가로 100주의 주식을 주당 3000원에 발행하기로 하였으며, 신주 발행시에 직접적인 거래원가가 10,000원이 발생하였다. 해당 거래에서 ①증가한 자본의 금액은 얼마이며, 거래 직후 재무상태표의 ②주식발행초과금의 잔액과 ③주식할인발행차금의 잔액을 구하라.")

[human] (주)라비티는 전기에 액면가 5000원의 100주의 주식을 주당 6000원에 발행하여 설립되었다. 당기 2월 1일, (주)라비티는 추가로 100주의 주식을 주당 3000원에 발행하기로 하였으며, 신주 발행시에 직접적인 거래원가가 10,000원이 발생하였다. 해당 거래에서 ①증가한 자본의 금액은 얼마이며, 거래 직후 재무상태표의 ②주식발행초과금의 잔액과 ③주식할인발행차금의 잔액을 구하라.

[ai] [{'text': '이 문제를 단계별로 풀어보겠습니다.\n\n## 주어진 정보\n- **전기**: 액면가 5,000원, 100주를 주당 6,000원에 발행\n- **당기 2월 1일**: 100주를 주당 3,000원에 발행, 직접 거래원가 10,000원 발생\n\n## 해결 과정\n\n### 1단계: 전기 발행 내역\n- 자본금: 5,000원 × 100주 = 500,000원\n- 주식발행초과금: (6,000원 - 5,000원) × 100주 = 100,000원\n\n### 2단계: 당기 신주 발행 내역', 'type': 'text'}, {'id': 'toolu_01JrxAeszonAyaQpeSLyoGf2', 'caller': {'type': 'direct'}, 'input': {'expression': '5000 * 100'}, 'name': 'calculator', 'type': 'tool_use'}, {'id': 'toolu_01Nno1bUzJRm4btV5eWUq3aC', 'caller': {'type': 'direct'}, 'input': {'expression': '3000 * 100'}, 'name': 'calculator', 'type': 'tool_use'}, {'id': 'toolu_01DzWJb9T4uUqeCpRWuf4VV5', 'caller': {'type': 'direct'}, 'input': {'expression': '300000 - 10000'}, 'name': 'calculator', 'type': 'tool_use'}]

[to

## 5. Free Test

In [5]:
ask("회사는 20X1년 7월 1일부터 20X2년 6월 30일까지 웹사이트 이용 서비스를 제공하고, 이용료 1,000만원을 20X2년 6월 30일에 후불로 받기로 하였음. 회사나 고객이 중도 해지하면 이용기간에 해당하는 금액은 지급해야 함\n\n이 경우, 20X1년 말에 회사가 대가를 받을 권리를 수취채권으로 인식하는지, 아니면 계약자산으로 인식하는지?")

KeyboardInterrupt: 